# 06_02 A hidden layer: how two neurons do what one never could

One neuron cannot learn XOR. This notebook solves it twice: once by hand, with a hidden layer whose weights
you can read, and once by letting PyTorch learn them. On the way you will fix a network that trains
faithfully and learns nothing, and watch a network whose hidden units can never become different.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-06-a-neuron-from-scratch", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import numpy as np
import torch
import torch.nn as nn
from nlpcheck import ask, guess, reveal, check_06_02

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
XOR = [0, 1, 1, 0]
step = lambda z: (z > 0).astype(int)
print("PyTorch", torch.__version__)

## 1. Recall

**r3.** Why can a single perceptron never learn XOR? (a) not enough epochs, (b) no straight line separates
the two classes, (c) the learning rate is too small

**r4.** A neuron with weights 1 and 1 and bias -1.5 computes which function? (`and`, `or` or `xor`)

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. XOR by hand, with a hidden layer

XOR is "OR, but not both". So use two neurons first: one that computes OR and one that computes NAND
(not both). Then a third neuron fires when both of them do. The first two are a **hidden layer**: nobody
sees their outputs except the neuron after them.

In [ ]:
h1 = step(X @ np.array([1, 1]) - 0.5)     # OR
h2 = step(X @ np.array([-1, -1]) + 1.5)   # NAND: fires unless both inputs are 1
out = step(h1 + h2 - 1.5)                 # AND of the two hidden neurons
for x, a, b, y in zip(X, h1, h2, out):
    print(f"input {x}  hidden ({a}, {b})  ->  {y}")

Read the hidden column. The inputs (0,1) and (1,0), which sat on opposite corners, both become (1, 1); the
other two become (1, 0) and (0, 1). In the hidden layer's coordinates the problem **is** separable, and the
last neuron draws its one straight line there. That is what a hidden layer does: it moves the points into a
space where a line works.

## 3. Letting PyTorch learn the weights, and the planted bug

**PyTorch** is the library most neural networks are written in today. `nn.Linear(2, 4)` is a layer of 4
neurons, each with 2 weights and a bias; `nn.Sequential` chains layers; the optimiser adjusts every weight to
lower the loss (how it does that is Lab 07). The network below has a hidden layer of 4 neurons. Predict:
after 2,000 steps of training, will it predict XOR, `[0, 1, 1, 0]`? (`yes` or `no`)

In [ ]:
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(XOR, dtype=torch.float32).unsqueeze(1)

def train(model, steps=2000, lr=0.05):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()      # the loss for a yes/no output; section 5 of 06_03 explains it
    for _ in range(steps):
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()
    predictions = (model(Xt) > 0).int().squeeze().tolist()
    return predictions, loss.item()

guess("learns_xor", None)   # "yes" or "no" 

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 4), nn.Linear(4, 1))   # the planted bug: see below
predictions, loss = train(model)
print("predictions", predictions, " loss", round(loss, 4))
reveal("learns_xor", "yes" if predictions == XOR else "no")

No. It gets two of the four wrong, and its loss sits at 0.6931, which is exactly ln 2: the loss of a coin
toss, a model that has learned nothing about which answer is right. It trained faithfully for 2,000 steps and learned nothing, because the hidden layer has no
**activation**. Two linear layers, W2(W1 x + b1) + b2, multiply out to a single linear layer W x + b, and a
single linear layer is the perceptron's straight line again. You can check the algebra with the network's own
numbers:

In [ ]:
A, B = [m for m in model if isinstance(m, nn.Linear)]   # the two linear layers, whatever sits between them
W = B.weight @ A.weight
b = B.weight @ A.bias + B.bias
print("largest difference between the network and one linear layer:", (model(Xt) - (Xt @ W.T + b)).abs().max().item())

A difference of about one hundred-millionth: rounding error. **Fix the planted bug**: in the cell two above,
put `nn.Tanh()` between the two `nn.Linear` layers, run it again, and run this check cell again. `Tanh` bends
each neuron's output into an S shape between -1 and 1, and a bend is all it takes. When it is fixed the
network predicts `[0, 1, 1, 0]` and the loss falls to about 0.00005. Run the check cell again too: the
difference is now large, because the network is no longer one straight line in disguise.

Try one more thing once it works: change `torch.manual_seed(0)` to `torch.manual_seed(1)` and run it again.
With that starting point the same network gets stuck at `[0, 0, 1, 1]`: training can settle in a poor
solution, which is one reason experiments fix their random seed. Set it back to 0 before you go on.

## 4. Where the weights start

PyTorch starts every weight at a small random number. Predict: if every weight instead starts at the **same**
value (here 0.5), will the four hidden neurons end training different from each other, or identical?
(`different` or `identical`)

In [ ]:
guess("constant_start", None)   # "different" or "identical" 

In [ ]:
torch.manual_seed(0)
same = nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1))
for p in same.parameters():
    nn.init.constant_(p, 0.5)            # every weight and bias starts at 0.5
predictions_same, loss_same = train(same)
rows = same[0].weight.data
identical = bool(torch.allclose(rows, rows[0].expand_as(rows)))
print(rows)
print("predictions", predictions_same, " loss", round(loss_same, 4))
reveal("constant_start", "identical" if identical else "different")

Identical. All four rows moved a long way (each weight is now about -6.7) and they moved together, because
neurons that start the same receive the same gradient and take the same step, every step. Four copies of one
neuron are one neuron, so the network is as weak as the perceptron again. Starting every weight at zero is
worse still: nothing moves at all, and the loss stays at 0.6931. **Random** starting weights break the
symmetry. PyTorch's default starts each weight at random within plus or minus one over the square root of the
layer's number of inputs, a rule from the family He and colleagues described in 2015, sized so that signals
neither vanish nor explode as they pass through the layers.

The book compares starting points on a harder data set, two rings of points. This cell repeats the
comparison. It trains four small networks, which takes under half a minute.

In [ ]:
from sklearn.datasets import make_circles
Xc, yc = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=1)
Xc = torch.tensor(Xc, dtype=torch.float32); yc = torch.tensor(yc, dtype=torch.float32).unsqueeze(1)

def rings(start):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(2, 10), nn.ReLU(), nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 1))
    for m in net:
        if isinstance(m, nn.Linear):
            if start == "zeros": nn.init.zeros_(m.weight)
            if start == "tiny random": nn.init.normal_(m.weight, std=0.01)
            if start == "large random": nn.init.normal_(m.weight, std=10)
            nn.init.zeros_(m.bias)
    opt = torch.optim.SGD(net.parameters(), lr=0.1); loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(1000):
        opt.zero_grad(); loss = loss_fn(net(Xc), yc); loss.backward(); opt.step()
    return loss.item(), ((net(Xc) > 0).float() == yc).float().mean().item()

for start in ("zeros", "tiny random", "large random", "PyTorch default"):
    loss_r, acc_r = rings(start)
    print(f"{start:16} loss {loss_r:.4f}  accuracy {acc_r:.0%}")

All zeros never leaves the coin toss: 50 percent, loss 0.6931. A tiny random start is hardly better, because
signals shrink layer by layer until almost nothing reaches the output (64 percent, the loss still 0.6931). A
large random start learns, clumsily (89 percent). PyTorch's default, scaled to each layer, separates the rings
(100 percent).

## 5. Save and check

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump({"layers": [str(m) for m in model], "predictions": predictions, "loss": loss,
           "identical_hidden_rows": identical}, open("out/06_02_xor.json", "w"), indent=1)
check_06_02()

Explain it back: why did adding one `nn.Tanh()` turn a network that could not learn XOR into one that could?

*Your explanation:* 